First check what files I have loaded by displaying the folder/schema path

In [0]:
# display(dbutils.fs.ls("/Volumes/workspace/fitbit_project/fitbitdata_csvuploads/"))

Then make tables from the uploaded csv files to be able to query and use SQL

In [0]:
# Map each raw filename to a clean table name
file_to_table = {
    "dailyActivity_merged.csv": "activity1",
    "dailyActivity_merged_4.12-5.12.16.csv": "activity2",
    "heartrate_seconds_merged.csv": "heartrate1",
    "heartrate_seconds_merged__4.12-5.12.16.csv": "heartrate2",
    "hourlySteps_merged.csv": "steps1",
    "hourlySteps_merged__4.12-5.12.16.csv": "steps2",
    "sleepDay_merged_4.12-5.12.16.csv": "sleepData",
}

base_path = "/Volumes/workspace/fitbit_project/fitbitdata_csvuploads/"

for file_name, table_name in file_to_table.items():
    # get the file path using the base and the file name
    file_path = base_path + file_name
    print(f"Loading {file_name} -> table: {table_name}")
    # Drop table first to avoid schema mismatch errors (bc I loaded the wrong table for sleep2 initially)
    spark.sql(f"DROP TABLE IF EXISTS workspace.fitbit_project.sleepdata")
    
    df = spark.read.csv(file_path, header=True, inferSchema=True)
    df.write.mode("overwrite").saveAsTable(f"workspace.fitbit_project.{table_name}")

print("Done.")


Loading sleepDay_merged_4.12-5.12.16.csv -> table: sleepData
Done.


In [0]:
%sql
SHOW CATALOGS;
SHOW SCHEMAS IN workspace;
SHOW TABLES IN workspace.fitbit_project;

-- clean up tables don't need from testing
-- DROP TABLE fitbit_project.sleep1;
-- DROP TABLE fitbit_project.sleep_combined;


database,tableName,isTemporary
fitbit_project,activity1,false
fitbit_project,activity2,false
fitbit_project,activity_combined,false
fitbit_project,heartrate1,false
fitbit_project,heartrate2,false
fitbit_project,heartrate_combined,false
fitbit_project,sleepdata,false
fitbit_project,steps1,false
fitbit_project,steps2,false
fitbit_project,steps_combined,false


In [0]:
df1 = spark.read.table("workspace.fitbit_project.activity1")
df2 = spark.read.table("workspace.fitbit_project.activity2")

print(df1.columns == df2.columns)
# the tables have the same column names in the same order (if returns Trure)

True


In [0]:
%sql
--DESCRIBE workspace.fitbit_project.activity1;
--DESCRIBE workspace.fitbit_project.activity2;

-- look at the data types and column names
DESCRIBE workspace.fitbit_project.activity_combined;

col_name,data_type,comment
Id,bigint,null
ActivityDate,date,null
TotalSteps,int,null
TotalDistance,double,null
TrackerDistance,double,null
LoggedActivitiesDistance,double,null
VeryActiveDistance,double,null
ModeratelyActiveDistance,double,null
LightActiveDistance,double,null
SedentaryActiveDistance,double,null


In [0]:
# now for each category, combine the two timepoint tables into 1 table
categories = ["activity", "heartrate", "steps"]

for cat in categories:
    df1 = spark.read.table(f"workspace.fitbit_project.{cat}1")
    df2 = spark.read.table(f"workspace.fitbit_project.{cat}2")    

    # if the columns match, combine the two tables
    if df1.columns == df2.columns:
        query = f"""
        CREATE OR REPLACE TABLE workspace.fitbit_project.{cat}_combined AS
        SELECT * FROM workspace.fitbit_project.{cat}1
        UNION
        SELECT * FROM workspace.fitbit_project.{cat}2
        """
        print(f"Running union for: {cat}")
        spark.sql(query)
    # if the columns don't match will want to investigate strucure of tables
    else:
        print(f"Columns don't match for: {cat}")
        print(df1.columns)
        print(df2.columns)

print("Done.")

Running union for: activity
Running union for: sleep
Running union for: heartrate
Running union for: steps
Done.


In [0]:
%sql
--check that the tables combined properly with the union (vertical cat)
SELECT COUNT(*) FROM workspace.fitbit_project.activity1;
SELECT COUNT(*) FROM workspace.fitbit_project.activity2;
SELECT COUNT(*) FROM workspace.fitbit_project.activity_combined;

-- check the dates actually span 3/2016 - 5/2016
SELECT MIN(ActivityDate), MAX(ActivityDate)
FROM workspace.fitbit_project.activity_combined;

--SleepDay is a string so convert that to a date
CREATE OR REPLACE TABLE workspace.fitbit_project.sleepdata AS
SELECT 
    Id,
    to_date(SleepDay, 'M/d/yyyy h:mm:ss a') AS SleepDay,
    TotalSleepRecords,
    TotalMinutesAsleep,
    TotalTimeInBed
FROM workspace.fitbit_project.sleepdata;

DESCRIBE workspace.fitbit_project.sleepdata;

SELECT SleepDay 
FROM workspace.fitbit_project.sleepdata 
LIMIT 10;

-- DESCRIBE workspace.fitbit_project.activity_combined


MIN(ActivityDate),MAX(ActivityDate)
2016-03-12,2016-05-12


In [0]:
%sql
--Now combine all physiological data into a single table to upload to Snowflake--

CREATE OR REPLACE TABLE fitbit_merged AS
SELECT
    a.Id,
    a.ActivityDate AS Date,
    a.TotalSteps,
    a.TotalDistance,
    a.Calories,
    a.VeryActiveMinutes,
    a.FairlyActiveMinutes,
    a.lightlyActiveMinutes,
    a.SedentaryMinutes,
    s.TotalMinutesAsleep,
    s.TotalTimeInBed
FROM workspace.fitbit_project.activity_combined a
LEFT JOIN workspace.fitbit_project.sleepdata s
    ON a.Id = s.Id AND a.ActivityDate = s.SleepDay;

-- DESCRIBE fitbit_merged;
SELECT * FROM fitbit_merged LIMIT 10;
SELECT COUNT(*) FROM fitbit_merged

COUNT(*)
1400


In [0]:
%sql
-- There are a bunch of Nulls so take care of those and check for any duplicates?
-- DESCRIBE fitbit_merged;
SELECT *
FROM fitbit_merged
WHERE TotalSteps IS NOT NULL
  AND Calories > 0

Id,Date,TotalSteps,TotalDistance,Calories,VeryActiveMinutes,FairlyActiveMinutes,lightlyActiveMinutes,SedentaryMinutes,TotalMinutesAsleep,TotalTimeInBed
1503960366,2016-04-02,11248,7.25,1843,40,11,244,636,null,null
1624580081,2016-03-31,4506,2.9300000667572,1498,7,4,144,1285,null,null
1644430081,2016-04-10,1329,0.970000028610229,489,0,0,35,207,null,null
1844505072,2016-04-09,4979,3.28999996185303,1807,0,0,184,620,null,null
1927972279,2016-04-11,1209,0.839999973773956,2255,0,0,73,842,null,null
2026352035,2016-04-10,5142,3.19000005722046,1515,0,0,230,654,null,null
2320127002,2016-04-06,6999,4.71999979019165,1950,0,0,320,1120,null,null
2320127002,2016-04-08,3417,2.29999995231628,1625,0,0,153,1287,null,null
2347167796,2016-04-06,11107,7.34000015258789,2058,14,46,196,759,null,null
2347167796,2016-04-08,10209,6.75,2104,2,6,316,711,null,null


In [0]:
%sql
--Now that the data is combined, get a summary of the data by week for each patient
CREATE OR REPLACE TABLE fitbit_weekly_summary AS
SELECT
    Id,
    WEEKOFYEAR(Date) AS Week,
    AVG(TotalSteps) AS AvgSteps,
    AVG(TotalDistance) AS AvgDistance,
    AVG(Calories) AS AvgCalories,
    AVG(VeryActiveMinutes) AS AvgActiveMins,
    AVG(FairlyActiveMinutes) AS AvgModeratelyActiveMins,
    AVG(lightlyActiveMinutes) AS AvgLightlyActiveMins,
    AVG(SedentaryMinutes) AS AvgSedentaryMins,
    AVG(TotalMinutesAsleep) AS AvgSleepMinutes,
    AVG(TotalMinutesAsleep) AS AvgMinInBed
FROM fitbit_merged
GROUP BY Id, WEEKOFYEAR(Date);

col_name,data_type,comment
Id,bigint,null
Week,int,null
AvgSteps,double,null
AvgDistance,double,null
AvgCalories,double,null
AvgActiveMins,double,null
AvgModeratelyActiveMins,double,null
AvgLightlyActiveMins,double,null
AvgSedentaryMins,double,null
AvgSleepMinutes,double,null


In [0]:
%sql
SELECT 
    CAST(Id AS STRING) AS PatientId,
    Week,
    AvgActiveMins,
    AvgSedentaryMins,
    AvgMinInBed,
    AvgSleepMinutes
FROM fitbit_weekly_summary
ORDER BY Week;


PatientId,Week,AvgActiveMins,AvgSedentaryMins,AvgMinInBed,AvgSleepMinutes
4057192912,10,0.0,1440.0,null,null
4020332650,10,0.0,764.0,null,null
4020332650,11,3.5714285714285716,1112.0,null,null
4057192912,11,4.285714285714286,1404.142857142857,null,null
4057192912,12,1.0,1367.7142857142858,null,null
1503960366,12,59.333333333333336,665.6666666666666,null,null
1624580081,12,0.0,1346.6666666666667,null,null
4020332650,12,0.0,1440.0,null,null
6391747486,13,15.333333333333334,1424.6666666666667,null,null
4558609924,13,0.3333333333333333,1082.6666666666667,null,null


Databricks visualization. Run in Databricks to view.

In [0]:
df = spark.read.table("fitbit_weekly_summary")
display(df)

Id,Week,AvgSteps,AvgDistance,AvgCalories,AvgActiveMins,AvgModeratelyActiveMins,AvgLightlyActiveMins,AvgSedentaryMins,AvgSleepMinutes,AvgMinInBed
1624580081,13,3331.714285714286,2.165714263916016,1396.142857142857,1.0,0.5714285714285714,122.14285714285714,1316.2857142857142,null,null
1644430081,14,7851.857142857143,5.708571434020996,2625.1428571428573,7.857142857142857,37.42857142857143,151.57142857142858,1072.0,null,null
2873212765,13,3581.3333333333335,2.406666715939839,1579.6666666666667,1.0,5.0,184.33333333333334,1249.6666666666667,null,null
4020332650,11,6678.857142857143,4.794285672051567,3122.0,3.5714285714285716,3.4285714285714284,102.28571428571429,1112.0,null,null
8378563200,13,8896.333333333334,7.053333361943577,3225.0,32.666666666666664,11.333333333333334,194.0,658.6666666666666,null,null
3977333714,13,7875.0,5.269999901453657,1469.3333333333333,9.666666666666666,9.333333333333334,250.66666666666666,810.6666666666666,null,null
7086361926,15,5192.625,3.5012499701697384,2106.125,25.125,13.375,97.375,854.125,465.6,465.6
8583815059,13,2682.6666666666665,2.0966666936874403,2425.6666666666665,0.0,0.0,136.33333333333334,1303.6666666666667,null,null
8792009665,13,6760.666666666667,4.326666673024497,2643.6666666666665,5.0,12.666666666666666,261.0,821.0,null,null
4020332650,14,7244.142857142857,5.19285716329302,3212.0,5.428571428571429,15.857142857142858,219.0,1084.142857142857,null,null
